In [ ]:
# import all libraries
import matplotlib.pyplot as plt
import numpy as np
from tensorflow.keras.layers import Dense
from tensorflow.keras.models import Sequential
from sklearn.metrics import mean_squared_error
from tqdm.keras import TqdmCallback
import gc

gc.collect()

In [ ]:
for Ki in range(48, 64, 16):
    # FFT size and training parameters
    N = 128
    K = Ki
    epo = 2000
    batch = 10000
    
    # Generate random input data and desired output data
    sig = np.random.randn(batch, N)
    F = np.fft.fft(sig, axis=-1)
    
    # First half of inputs/outputs is real part, second half is imaginary part
    X = sig
    Y = F.real
    
    # Create model with single hidden layers, same number of outputs as inputs.
    # No bias needed.  No activation function, since DFT is linear.
    model = Sequential([
        Dense(K, input_dim=N, use_bias=False),  # 降维层
        Dense(N, use_bias=False)                # 恢复层
    ])
    model.compile(loss='mean_squared_error', optimizer='adam')
    model.fit(X, Y, epochs=epo, batch_size=batch, validation_split=0.2, verbose=0, callbacks=[TqdmCallback(verbose=1)])
    # Confirm that it works
    # data = np.arange(N)
    data = np.random.randn(N)
    
    # Prediction
    def ANN_DFT(x):
        if len(x) != N:
            raise ValueError(f'Input must be length {N}')
        pred = model.predict(x[np.newaxis])[0]
        return pred
    
    ANN = ANN_DFT(data)
    FFT = np.fft.fft(data)
    
    print(mean_squared_error(ANN, FFT.real))
    print(f'ANN matches FFT (real parts): {np.allclose(ANN, FFT.real)}')
    
    # Save the NumPy array to a text file with space as the delimiter
    # np.savetxt('outtxt/FFT_NN_'+str(N)+'_'+str(K)+'.txt', ANN, delimiter=' ')
    # np.savetxt('outtxt/FFT_PY_'+str(N)+'.txt', FFT.real, delimiter=' ')
    # Read the text file back into a NumPy array
    # loaded_array = np.loadtxt(text_file_path, delimiter=' ')
    # Save the model
    model.save(f"model/1D_FFT_{N}_reduce_{K}.h5")